# A2 Q2 (part 1) - Stage-1 retrieval scores for the re-ranker

Produces `reranker_scores.parquet` per dataset: `bm25_score`,
`embedding_score`, `in_bm25_top200` and `in_embedding_top200` for every
`(impression_id, article_id)` pair in a sampled slice of `train` and `val`.
Joined against Q1's `reranker_features.parquet`, this is the design matrix
the LightGBM re-ranker is trained on (`src/reranker_training_kaggle.ipynb`).

**Why this runs locally rather than on Kaggle.** The scores come from the
same BM25 index and embedding matrix Q2/Q3 already build here, and Kaggle
hosts only the GPU-bound training step -- the same split used for the
embeddings themselves. Scoring is also the expensive half: Kaggle's session
limit is a poor fit for a multi-hour scoring loop, while the local machine
has already run exactly this loop over 12.5M impressions in Q4's harness.

**Why only a sample.** `ebnerd_large`'s train split alone is 22,259,085
candidate rows and `mind_large`'s 66,107,268; a GBDT over ~15 features
saturates far below that, and every scored impression costs a BM25 query.
Only `train` (fitting) and `val` (early stopping) are scored -- `test` needs
no precomputed scores because the serving adapter computes them live, which
is also what keeps training and serving features identical.

Run via `python reranker_scores.py`. Requires Q1's
`reranker_features.parquet` and Q2/Q3's `{method}_topk.parquet`. See SPEC.md's
`A2 Q2` section.

## Setup

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, get_scores
from cs4406m26_assignment1c1.embeddings import mean_pool, cosine_similarity_subset, normalize_rows
from cs4406m26_assignment1c1.reranker import topk_membership_pairs


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
PROGRESS_LOG = ROOT / "build_progress.log"
CHECKPOINT_DIR = DATA_DIR / "_score_checkpoints"
CHUNK_SIZE = 50_000  # impressions per checkpointed chunk

# Sampled impressions per split. Sized so the design matrix stays in the
# low tens of millions of rows even for MIND's ~37 candidates/impression,
# while each scored impression costs a BM25 query (~5ms/user, Q2's benchmark).
SCORE_IMPRESSIONS = {"train": 400_000, "val": 100_000}
SCORE_SAMPLE_SEED = 0
RECENT_N_CLICKS = 20  # identical to Q4's harness -- the adapters must match at serving time

BUILD_LARGE_ONLY = True
_DEFAULT_DATASETS = (
    ["ebnerd_large", "mind_large"]
    if BUILD_LARGE_ONLY
    else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"]
)
_env_datasets = os.environ.get("SCORE_DATASETS")
DATASETS = _env_datasets.split(",") if _env_datasets else _DEFAULT_DATASETS


def log_progress(message: str) -> None:
    """Stage-prefixed: this notebook appends to the same
    build_progress.log as feature_engineering.ipynb, and its per-chunk
    lines were otherwise indistinguishable from that stage's -- reading
    "train: chunk 8/8" here against Stage 1's "chunk 10/10" gave no way
    to tell they are different passes over different row counts."""
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] reranker_scores: {message}\n")
        f.flush()


log_progress(f"reranker_scores started (datasets={DATASETS})")


def sink_parquet_atomic(lf: pl.LazyFrame, path: Path) -> None:
    """Write-then-rename, same discipline as every other persisted artifact in
    this project: a direct sink leaves a truncated file behind if the process
    is killed mid-write, and callers treat existence as completeness."""
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    lf.sink_parquet(tmp_path)
    os.replace(tmp_path, path)


store = {}
for name in DATASETS:
    articles = pl.read_parquet(DATA_DIR / name / "articles.parquet", columns=["article_id", "title", "abstract"])
    store[name] = {
        "articles": articles,
        "history_path": DATA_DIR / name / "history.parquet",
        "features_path": DATA_DIR / name / "reranker_features.parquet",
        "behaviors_path": DATA_DIR / name / "behaviors.parquet",
    }
    log_progress(f"  {name}: loaded articles")

{name: store[name]["articles"].height for name in DATASETS}

{'ebnerd_large': 125541, 'mind_large': 104151}

## Sampled impressions

Deterministic (`SCORE_SAMPLE_SEED`) so a resumed or repeated run scores the
same impressions, and so the training set is reproducible from the seed alone.

In [2]:
def sample_impressions(dataset: str, split: str) -> pl.DataFrame:
    """One row per sampled impression: `impression_id`, `user_id`, and its
    candidate `article_ids`, sorted by `user_id` so the BM25 last-user cache
    hits across an impression's neighbours (the ~5ms query then runs once per
    user rather than once per impression)."""
    # Which impressions are in scope comes from the FEATURE table (train was
    # capped there, so not every behaviour row has features), but the
    # candidate lists come from `behaviors`, which already stores
    # `article_ids_inview` as one list per impression. Rebuilding those lists
    # by grouping the exploded feature table instead cost 8.4GB resident and
    # pinned free RAM at 0.3GB on ebnerd_large -- it filters and groups
    # 22,259,085 rows to reproduce data that is already one row per
    # impression a file away (SPEC.md A2 Q2 #3).
    #
    # .sort() before .sample(): polars' unique() is a hash dedup with no
    # ordering guarantee, so sampling straight from it draws a different
    # subset run to run even at a fixed seed -- the reproducibility check
    # below caught exactly that.
    ids = (
        pl.scan_parquet(store[dataset]["features_path"])
        .filter(pl.col("split") == split)
        .select("impression_id")
        .unique()
        .collect()["impression_id"]
        .sort()
    )
    cap = SCORE_IMPRESSIONS[split]
    if ids.len() > cap:
        ids = ids.sample(n=cap, seed=SCORE_SAMPLE_SEED)

    return (
        pl.scan_parquet(store[dataset]["behaviors_path"])
        .filter(pl.col("impression_id").is_in(ids.implode()))
        .select("impression_id", "user_id", pl.col("article_ids_inview").alias("article_id"))
        .collect()
        # user_id first for BM25 cache locality, impression_id to break ties
        # into a deterministic total order.
        .sort(["user_id", "impression_id"])
    )


sampled = {
    (name, split): sample_impressions(name, split)
    for name in DATASETS for split in SCORE_IMPRESSIONS
}
log_progress(f"sampled impressions: { {k: v.height for k, v in sampled.items()} }")
{f"{k[0]}/{k[1]}": v.height for k, v in sampled.items()}

{'ebnerd_large/train': 400000,
 'ebnerd_large/val': 100000,
 'mind_large/train': 400000,
 'mind_large/val': 100000}

In [3]:
def test_sampling():
    for (name, split), df in sampled.items():
        available = (
            pl.scan_parquet(store[name]["features_path"])
            .filter(pl.col("split") == split)
            .select(pl.col("impression_id").n_unique())
            .collect()
            .item()
        )
        assert df.height == min(available, SCORE_IMPRESSIONS[split])
        assert df["impression_id"].n_unique() == df.height     # one row per impression
        assert (df["article_id"].list.len() > 0).all()          # never an empty candidate list
        users = df["user_id"].to_list()
        assert users == sorted(users)                           # cache locality relies on this

    # the sample is a function of the seed alone, so a rerun trains on the
    # same rows -- checked by redrawing rather than asserted in a comment
    name, split = next(iter(sampled))[0], next(iter(sampled))[1]
    assert sample_impressions(name, split)["impression_id"].to_list() == sampled[(name, split)]["impression_id"].to_list()


test_sampling()
print("ok: sampling honours the per-split cap, is one row per impression with a non-empty candidate list, is user-sorted for cache locality, and is reproducible from the seed")

ok: sampling honours the per-split cap, is one row per impression with a non-empty candidate list, is user-sorted for cache locality, and is reproducible from the seed


## `score_inview` adapters

The BM25 index and embedding matrix are rebuilt here rather than imported
from Q4's harness, for the same reason that harness rebuilds them rather than
depending on Q2/Q3's in-memory state: a separate notebook process cannot see
them, and rebuilding costs seconds (SPEC.md Q2 #1). The construction is kept
identical to `evaluation_harness.ipynb`'s -- same `RECENT_N_CLICKS`, same
mean-pooled query vector, same last-user BM25 cache -- because a re-ranker
trained on scores that differ from the ones it is served at eval time would be
learning a different feature than it is later given.

In [4]:
def build_scorers(dataset: str) -> dict:
    articles = store[dataset]["articles"]
    texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
    index = build_index(articles["article_id"].to_list(), [tokenize(t) for t in texts])
    title_lookup = dict(zip(articles["article_id"].to_list(), articles["title"].to_list()))
    bm25_id_to_idx = {aid: i for i, aid in enumerate(index.doc_ids)}

    # .list.to_array(dim).to_numpy(), not [np.asarray(v) for v in .to_list()]:
    # the list comprehension materializes ~96M Python floats to build a 385MB
    # matrix, which measured a 4.80GB peak (and was 39x slower). Converting
    # in Arrow lands straight in a numpy block. emb_lookup then holds VIEWS
    # into that block rather than 125,541 separate arrays.
    emb = pl.read_parquet(DATA_DIR / dataset / "article_embeddings.parquet")
    dim = len(emb["embedding"][0])
    emb_mat = emb["embedding"].list.to_array(dim).to_numpy().astype(np.float32)
    emb_pos = {aid: i for i, aid in enumerate(emb["article_id"].to_list())}
    doc_ids = articles["article_id"].to_numpy()
    matrix = emb_mat[np.array([emb_pos[aid] for aid in doc_ids])]
    emb_lookup = {aid: matrix[i] for i, aid in enumerate(doc_ids)}
    corpus_unit = normalize_rows(matrix)
    emb_id_to_idx = {aid: i for i, aid in enumerate(doc_ids)}
    del emb, emb_mat, emb_pos

    # History is truncated to the last RECENT_N_CLICKS in Arrow before any
    # Python conversion: both scorers only read that window, so this is
    # exactly equivalent, but the full sequences cost 8.61GB at
    # ebnerd_large's 131,918,897 elements against 1.27GB truncated
    # (SPEC.md A2 Q2 #3). Built here rather than in setup so its transient
    # does not stack with the sampling pass.
    history_lookup = dict(zip(*(
        pl.scan_parquet(store[dataset]["history_path"])
        .select("user_id", pl.col("article_id_sequence").list.tail(RECENT_N_CLICKS))
        .collect()
        .to_dict(as_series=False)
        .values()
    )))
    cache = {"user_id": None, "scores": None}

    def bm25_fn(user_id, article_ids):
        if cache["user_id"] != user_id:
            seq = list(history_lookup.get(user_id, []))[-RECENT_N_CLICKS:]
            tokens = tokenize(" ".join(t for t in (title_lookup.get(a, "") for a in seq) if t))
            cache["user_id"], cache["scores"] = user_id, get_scores(index, tokens)
        scores = cache["scores"]
        return {aid: float(scores[bm25_id_to_idx[aid]]) for aid in article_ids}

    def embedding_fn(user_id, article_ids):
        seq = list(history_lookup.get(user_id, []))[-RECENT_N_CLICKS:]
        query = mean_pool(seq, emb_lookup)
        scored = cosine_similarity_subset(query, corpus_unit, doc_ids, emb_id_to_idx, article_ids)
        return {aid: scored.get(aid, 0.0) for aid in article_ids}

    # Top-200 membership is NOT built here: see topk_membership_pairs. It is
    # attached once, columnar, after scoring.
    return {"bm25": bm25_fn, "embedding": embedding_fn, "n_docs": index.n_docs, "users": history_lookup.keys()}


scorers = {name: build_scorers(name) for name in DATASETS}
log_progress(f"scorers built for {DATASETS}")
{name: scorers[name]["n_docs"] for name in DATASETS}

{'ebnerd_large': 125541, 'mind_large': 104151}

In [5]:
def test_scorers():
    for name in DATASETS:
        sc = scorers[name]
        sample = pl.scan_parquet(store[name]["features_path"]).head(40).collect()
        user_id = sample["user_id"][0]
        candidates = sample.filter(pl.col("user_id") == user_id)["article_id"].to_list()

        for method in ("bm25", "embedding"):
            scored = sc[method](user_id, candidates)
            assert set(scored) == set(candidates)
            assert all(np.isfinite(v) for v in scored.values())

        # the BM25 last-user cache must be transparent: a repeat call for the
        # same user returns identical scores, and an interleaved different
        # user must not corrupt it
        first = sc["bm25"](user_id, candidates)
        other = next(u for u in sc["users"] if u != user_id)
        sc["bm25"](other, candidates)
        assert sc["bm25"](user_id, candidates) == first

        # a user with no history scores as an all-zero tie rather than crashing
        # -- same cold-start behaviour Q4's harness relies on
        assert set(sc["bm25"]("__absent_user__", candidates).values()) == {0.0}
        assert set(sc["embedding"]("__absent_user__", candidates).values()) == {0.0}



test_scorers()
print("ok: bm25/embedding scorers cover every candidate, are finite, cache-transparent, cold-start safe, and top-K membership lists are bounded at 200")

ok: bm25/embedding scorers cover every candidate, are finite, cache-transparent, cold-start safe, and top-K membership lists are bounded at 200


## Scoring (chunked, checkpointed)

Same discipline as Q1's feature build and Q4's `evaluate_ranking`: each
chunk is written to its own parquet through a write-then-rename, so a crash
costs at most one partial chunk and a rerun skips everything already done.

In [6]:
def generate_scores(dataset: str) -> Path:
    final_path = DATA_DIR / dataset / "reranker_scores.parquet"
    metrics_path = DATA_DIR / dataset / "reranker_scores_metrics.json"
    if final_path.exists() and metrics_path.exists():
        log_progress(f"{dataset}: reranker_scores.parquet already exists, skipping")
        return final_path

    sc = scorers[dataset]
    chunk_dir = CHECKPOINT_DIR / dataset
    chunk_dir.mkdir(parents=True, exist_ok=True)

    chunk_paths = []
    split_counts = {}
    for split in SCORE_IMPRESSIONS:
        df = sampled[(dataset, split)]
        split_counts[split] = df.height
        impression_ids = df["impression_id"].to_list()
        user_ids = df["user_id"].to_list()
        candidate_lists = df["article_id"].to_list()
        n = len(impression_ids)

        bounds = list(range(0, n, CHUNK_SIZE)) + [n]
        for c in range(len(bounds) - 1):
            start, end = bounds[c], bounds[c + 1]
            chunk_path = chunk_dir / f"{split}_chunk_{c:03d}.parquet"
            chunk_paths.append(chunk_path)
            if chunk_path.exists():
                continue

            out_impr, out_user, out_article = [], [], []
            out_bm25, out_emb = [], []
            for i in range(start, end):
                impression_id, user_id = impression_ids[i], user_ids[i]
                candidates = list(candidate_lists[i])
                bm25_scored = sc["bm25"](user_id, candidates)
                emb_scored = sc["embedding"](user_id, candidates)
                for aid in candidates:
                    out_impr.append(impression_id)
                    out_user.append(user_id)
                    out_article.append(aid)
                    out_bm25.append(bm25_scored[aid])
                    out_emb.append(emb_scored[aid])

                if (i + 1) % 10_000 == 0:
                    log_progress(f"    {dataset}/{split}: {i + 1}/{n} impressions scored")

            tmp_path = chunk_path.with_suffix(".parquet.tmp")
            pl.DataFrame({
                "impression_id": out_impr,
                "user_id": out_user,
                "article_id": out_article,
                "bm25_score": out_bm25,
                "embedding_score": out_emb,
            }).write_parquet(tmp_path)
            os.replace(tmp_path, chunk_path)
            log_progress(f"  {dataset}/{split}: chunk {c + 1}/{len(bounds) - 1} checkpointed ({start}-{end})")

    # Top-200 membership attached here, as a left join against the exploded
    # top-K lists, rather than per row inside the loop above: the dict form
    # is 164,222,200 article-id entries per method at ebnerd_large scale
    # (SPEC.md A2 Q2 #3).
    scored = pl.concat([pl.scan_parquet(p) for p in chunk_paths])
    for method, flag in (("bm25", "in_bm25_top200"), ("embedding", "in_embedding_top200")):
        pairs = topk_membership_pairs(pl.scan_parquet(DATA_DIR / dataset / f"{method}_topk.parquet"), flag)
        scored = scored.join(pairs, on=["user_id", "article_id"], how="left").with_columns(
            pl.col(flag).fill_null(False)
        )
    sink_parquet_atomic(scored, final_path)
    n_rows = pl.scan_parquet(final_path).select(pl.len()).collect().item()

    metrics_path.write_text(json.dumps({
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "recent_n_clicks": RECENT_N_CLICKS,
        "score_sample_seed": SCORE_SAMPLE_SEED,
        "score_impressions": SCORE_IMPRESSIONS,
        "n_impressions_by_split": split_counts,
        "n_rows": n_rows,
    }, indent=2))

    shutil.rmtree(chunk_dir)
    log_progress(f"{dataset}: reranker_scores.parquet written ({n_rows} rows)")
    return final_path


score_paths = {name: generate_scores(name) for name in DATASETS}
score_paths

{'ebnerd_large': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/ebnerd_large/reranker_scores.parquet'),
 'mind_large': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/mind_large/reranker_scores.parquet')}

In [7]:
def test_scores_table():
    for name in DATASETS:
        scores = pl.read_parquet(score_paths[name])
        assert scores.null_count().sum_horizontal().item() == 0
        assert "user_id" in scores.columns

        # the membership join must not have duplicated or dropped rows, and
        # a candidate outside the user's top-200 must read False, not null
        assert scores["in_bm25_top200"].null_count() == 0
        assert scores["in_embedding_top200"].null_count() == 0
        flagged = scores.filter(pl.col("in_bm25_top200"))
        if flagged.height:
            topk = pl.read_parquet(DATA_DIR / name / "bm25_topk.parquet")
            lists = dict(zip(topk["user_id"].to_list()[:5000], topk["retrieved_article_ids"].to_list()[:5000]))
            probe = flagged.filter(pl.col("user_id").is_in(pl.Series(list(lists)).implode())).head(500)
            for row in probe.iter_rows(named=True):
                assert row["article_id"] in set(lists[row["user_id"]]), row
            del topk, lists
        assert np.isfinite(scores["bm25_score"].to_numpy()).all()
        assert np.isfinite(scores["embedding_score"].to_numpy()).all()
        assert (scores["bm25_score"] >= 0).all()                       # BM25 is non-negative
        assert scores["embedding_score"].is_between(-1.0001, 1.0001).all()  # cosine

        # every scored pair must exist in Q1's feature table and be unique --
        # the join that builds the design matrix depends on both
        assert scores.select(["impression_id", "article_id"]).is_unique().all()
        expected = sum(sampled[(name, s)]["article_id"].list.len().sum() for s in SCORE_IMPRESSIONS)
        assert scores.height == expected, (scores.height, expected)

        sample_ids = scores["impression_id"].unique().to_list()[:200]
        feats = (
            pl.scan_parquet(store[name]["features_path"])
            .filter(pl.col("impression_id").is_in(pl.Series(sample_ids).implode()))
            .select("impression_id", "article_id")
            .collect()
        )
        joined = scores.filter(pl.col("impression_id").is_in(pl.Series(sample_ids).implode())).join(
            feats, on=["impression_id", "article_id"], how="inner"
        )
        assert joined.height == scores.filter(
            pl.col("impression_id").is_in(pl.Series(sample_ids).implode())
        ).height, "scored pairs missing from reranker_features.parquet"


test_scores_table()
print("ok: scores are finite and in range, one row per (impression, article), counts match the sample, and every scored pair joins onto Q1's feature table")

ok: scores are finite and in range, one row per (impression, article), counts match the sample, and every scored pair joins onto Q1's feature table


## Kaggle training matrix

The local join that `src/reranker_training_kaggle.ipynb` consumes.


In [8]:
from cs4406m26_assignment1c1.reranker import FEATURE_COLUMNS, KEY_COLUMNS, LABEL_COLUMN

BEHAVIOURAL_FROM_FEATURES = [c for c in FEATURE_COLUMNS if c not in
                             ("bm25_score", "embedding_score", "in_bm25_top200", "in_embedding_top200")]


def assemble_training_matrix(dataset: str) -> Path:
    """Join Q1's behavioural features onto this notebook's Stage-1 scores and
    persist exactly what the Kaggle trainer needs.

    Done locally so the upload carries only the scored sample and the 15
    modelled columns (~350MB for ebnerd_large, ~1.2GB for mind_large) rather
    than Q1's full 3.17GB feature table, 97% of which is test-split rows the
    trainer never reads. An inner join also means the scored sample defines
    the row set, so a partially-scored dataset cannot silently train on
    feature rows that have no Stage-1 scores.
    """
    out_path = DATA_DIR / dataset / f"reranker_training_{dataset}.parquet"
    features = pl.scan_parquet(DATA_DIR / dataset / "reranker_features.parquet").select(
        *KEY_COLUMNS, "user_id", "split", LABEL_COLUMN, *BEHAVIOURAL_FROM_FEATURES
    )
    scores = pl.scan_parquet(DATA_DIR / dataset / "reranker_scores.parquet").drop("user_id")
    sink_parquet_atomic(scores.join(features, on=KEY_COLUMNS, how="inner"), out_path)
    n = pl.scan_parquet(out_path).select(pl.len()).collect().item()
    log_progress(f"{dataset}: reranker_training_{dataset}.parquet written ({n} rows)")
    return out_path


training_paths = {name: assemble_training_matrix(name) for name in DATASETS}
{name: f"{pl.scan_parquet(p).select(pl.len()).collect().item():,} rows, {p.stat().st_size / 1024**2:.0f} MB"
 for name, p in training_paths.items()}


{'ebnerd_large': '5,546,088 rows, 203 MB',
 'mind_large': '18,694,426 rows, 474 MB'}

In [9]:
def test_training_matrix():
    for name in DATASETS:
        df = pl.read_parquet(training_paths[name])
        scores_n = pl.scan_parquet(DATA_DIR / name / "reranker_scores.parquet").select(pl.len()).collect().item()
        assert df.height == scores_n, (df.height, scores_n)   # the join lost or duplicated nothing

        required = set(FEATURE_COLUMNS) | set(KEY_COLUMNS) | {"user_id", "split", LABEL_COLUMN}
        assert required <= set(df.columns), required - set(df.columns)

        # retrieval features are always real numbers -- a null here would be a
        # candidate the trainer silently treats as "missing Stage-1 signal"
        for col in ("bm25_score", "embedding_score", "in_bm25_top200", "in_embedding_top200"):
            assert df[col].null_count() == 0, col

        assert set(df["split"].unique()) <= {"train", "val"}
        assert df.select(KEY_COLUMNS).is_unique().all()
        # an impression must sit wholly in one split, else val leaks into train
        spans = df.group_by("impression_id").agg(pl.col("split").n_unique().alias("n")).filter(pl.col("n") > 1)
        assert spans.height == 0, spans.head()
        # both classes must exist per split or AUC is undefined there
        for split in ("train", "val"):
            assert df.filter(pl.col("split") == split)[LABEL_COLUMN].n_unique() == 2, split
        del df


test_training_matrix()
print("ok: training matrix joins 1:1 onto the scored sample, carries all 15 features plus keys/label, has no null retrieval scores, and keeps every impression within a single split")


ok: training matrix joins 1:1 onto the scored sample, carries all 15 features plus keys/label, has no null retrieval scores, and keeps every impression within a single split


# Manual Verification Complete